In [1]:
import pandas as pd

df = pd.read_csv("DDI_clean_with_SMILES_Level1.csv")

display(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

print("\nLevel1 distribution:")
print(df["Level1"].value_counts())

print("\nLevel1_binary distribution:")
print(df["Level1_binary"].value_counts())

print("\nInteraction_ID distribution:")
print(df["Interaction_ID"].value_counts())

print("\nNumber of Level2 classes:")
print(df["Interaction_ID"].nunique())

,Drug1,Drug2,Interaction_ID,SMILES1,SMILES2,Level1,Level1_binary
0,DB01115,DB08807,1,COC(=O)C1=C(C)NC(C)=C(C1C1=CC=CC=C1[N+]([O-])=...,CC1=CC2=C(N1)C=CC=C2OCC(CNC(C)(C)C)OC(=O)C1=CC...,3,1
1,DB01115,DB00187,1,COC(=O)C1=C(C)NC(C)=C(C1C1=CC=CC=C1[N+]([O-])=...,COC(=O)CCC1=CC=C(OCC(O)CNC(C)C)C=C1,3,1
2,DB01115,DB00871,1,COC(=O)C1=C(C)NC(C)=C(C1C1=CC=CC=C1[N+]([O-])=...,CC(C)(C)NCC(O)C1=CC(O)=CC(O)=C1,3,1
3,DB01115,DB01295,1,COC(=O)C1=C(C)NC(C)=C(C1C1=CC=CC=C1[N+]([O-])=...,COC1=C(OC)C=C(CCNCC(O)COC2=CC=CC(C)=C2)C=C1,3,1
4,DB01115,DB01297,1,COC(=O)C1=C(C)NC(C)=C(C1C1=CC=CC=C1[N+]([O-])=...,CC(C)NCC(O)COC1=CC=C(NC(C)=O)C=C1,3,1


Shape: (222127, 7)
Columns: ['Drug1', 'Drug2', 'Interaction_ID', 'SMILES1', 'SMILES2', 'Level1', 'Level1_binary']

Level1 distribution:
Level1
2    155183
3     66944
Name: count, dtype: int64

Level1_binary distribution:
Level1_binary
0    155183
1     66944
Name: count, dtype: int64

Interaction_ID distribution:
Interaction_ID
113    62996
89     37204
3      27258
20     11765
75      9937
       ...  
79        10
54         8
8          7
40         7
101        7
Name: count, Length: 113, dtype: int64

Number of Level2 classes:
113


In [2]:
import pandas as pd

df = pd.read_csv("DDI_clean_with_SMILES_Level1.csv")

class_counts = df["Interaction_ID"].value_counts()

print("Number of Level2 classes:", df["Interaction_ID"].nunique())
print("\nClass count summary:")
print(class_counts.describe())

print("\nTop 20 classes:")
display(class_counts.head(20))

print("\nBottom 30 classes:")
display(class_counts.tail(30))

Number of Level2 classes: 113

Class count summary:
count      113.000000
mean      1965.725664
std       7416.315243
min          7.000000
25%         21.000000
50%        140.000000
75%        803.000000
max      62996.000000
Name: count, dtype: float64

Top 20 classes:


Interaction_ID
113    62996
89     37204
3      27258
20     11765
75      9937
74      9729
53      5884
104     5342
34      5300
80      3813
60      3520
10      2819
81      2399
58      2379
47      2106
108     1825
13      1614
57      1503
95      1499
109     1452
Name: count, dtype: int64


Bottom 30 classes:


Interaction_ID
26     27
107    21
51     20
66     20
102    18
7      17
90     16
24     16
25     15
71     15
69     15
64     15
36     15
32     14
65     14
4      14
112    13
82     12
31     12
77     12
93     12
23     11
59     11
83     10
9      10
79     10
54      8
8       7
40      7
101     7
Name: count, dtype: int64

In [3]:
min_samples = 100

class_counts = df["Interaction_ID"].value_counts()
valid_classes = class_counts[class_counts >= min_samples].index

df_level2 = df[df["Interaction_ID"].isin(valid_classes)].copy()

print("Original data:", df.shape)
print("Filtered Level2 data:", df_level2.shape)
print("Retained Level2 classes:", df_level2["Interaction_ID"].nunique())

print("\nRemoved rare classes:", df["Interaction_ID"].nunique() - df_level2["Interaction_ID"].nunique())

print("\nClass distribution after filtering:")
display(df_level2["Interaction_ID"].value_counts())

df_level2.to_csv("DDI_Level2_filtered_min100.csv", index=False)
print("Saved: DDI_Level2_filtered_min100.csv")

Original data: (222127, 7)
Filtered Level2 data: (220434, 7)
Retained Level2 classes: 61

Removed rare classes: 52

Class distribution after filtering:


Interaction_ID
113    62996
89     37204
3      27258
20     11765
75      9937
       ...  
14       140
111      130
94       116
45       109
41       104
Name: count, Length: 61, dtype: int64

Saved: DDI_Level2_filtered_min100.csv


In [4]:
retained_percent = len(df_level2) / len(df) * 100
print("Retained percentage:", round(retained_percent, 2), "%")

Retained percentage: 99.24 %


In [8]:
import pickle
import numpy as np

with open("DDI_feature_artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

print(artifacts.keys())

drug_fps = artifacts["drug_fps"]
drug_enzymes = artifacts["drug_enzymes"]
drug_cyps = artifacts["drug_cyps"]
drug_roles = artifacts["drug_roles"]
drug_sideeffects = artifacts["drug_sideeffects"]
drug_atc = artifacts["drug_atc"]

dict_keys(['drug_fps', 'drug_enzymes', 'drug_cyps', 'drug_roles', 'drug_sideeffects', 'drug_atc'])


In [2]:
def jaccard(a, b):
    a = set(a) if a is not None else set()
    b = set(b) if b is not None else set()
    if len(a) == 0 and len(b) == 0:
        return 0.0
    return len(a & b) / len(a | b)

def shared_count(a, b):
    a = set(a) if a is not None else set()
    b = set(b) if b is not None else set()
    return len(a & b)

def get_set_feature(dictionary, drug_id):
    return dictionary.get(drug_id, set())

def get_atc_levels(drug_id):
    atc = drug_atc.get(drug_id, {})
    return [
        atc.get("niveau_1", None),
        atc.get("niveau_2", None),
        atc.get("niveau_3", None),
        atc.get("niveau_4", None)
    ]

def atc_match_features(d1, d2):
    atc1 = get_atc_levels(d1)
    atc2 = get_atc_levels(d2)
    return [1 if a == b and a is not None else 0 for a, b in zip(atc1, atc2)]

def build_pair_features_level2(row):
    d1 = row["Drug1"]
    d2 = row["Drug2"]

    fp1 = drug_fps.get(d1)
    fp2 = drug_fps.get(d2)

    if fp1 is None or fp2 is None:
        return None

    fp1 = np.array(fp1)
    fp2 = np.array(fp2)

    morgan_features = np.concatenate([fp1, fp2])

    enz1 = get_set_feature(drug_enzymes, d1)
    enz2 = get_set_feature(drug_enzymes, d2)

    cyp1 = get_set_feature(drug_cyps, d1)
    cyp2 = get_set_feature(drug_cyps, d2)

    role1 = get_set_feature(drug_roles, d1)
    role2 = get_set_feature(drug_roles, d2)

    se1 = get_set_feature(drug_sideeffects, d1)
    se2 = get_set_feature(drug_sideeffects, d2)

    extra_features = np.array([
        jaccard(enz1, enz2),
        shared_count(enz1, enz2),

        jaccard(cyp1, cyp2),
        shared_count(cyp1, cyp2),

        jaccard(role1, role2),

        jaccard(se1, se2),
        shared_count(se1, se2),

        *atc_match_features(d1, d2)
    ])

    return np.concatenate([morgan_features, extra_features])

In [7]:
import pandas as pd
import numpy as np
import pickle

# قراءة الملف الأساسي
df = pd.read_csv("DDI_clean_with_SMILES_Level1.csv")

# إعادة فلترة Level 2
min_samples = 100

class_counts = df["Interaction_ID"].value_counts()
valid_classes = class_counts[class_counts >= min_samples].index

df_level2 = df[df["Interaction_ID"].isin(valid_classes)].copy()

print("Original data:", df.shape)
print("Filtered Level2 data:", df_level2.shape)
print("Retained Level2 classes:", df_level2["Interaction_ID"].nunique())
print("Retained percentage:", round(len(df_level2) / len(df) * 100, 2), "%")

Original data: (222127, 7)
Filtered Level2 data: (220434, 7)
Retained Level2 classes: 61
Retained percentage: 99.24 %


In [8]:
max_per_class = 1000

df_level2_sample = (
    df_level2
    .groupby("Interaction_ID", group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), max_per_class), random_state=42))
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print(df_level2_sample.shape)
print(df_level2_sample["Interaction_ID"].nunique())
print(df_level2_sample["Interaction_ID"].value_counts().describe())

(39242, 7)
61
count      61.000000
mean      643.311475
std       355.391312
min       104.000000
25%       329.000000
50%       735.000000
75%      1000.000000
max      1000.000000
Name: count, dtype: float64


C:\Users\user\AppData\Local\Temp\ipykernel_18976\1648117242.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), max_per_class), random_state=42))


In [9]:
import pickle
import numpy as np
import pandas as pd
import gc

with open("DDI_feature_artifacts.pkl", "rb") as f:
    artifacts = pickle.load(f)

drug_fps = artifacts["drug_fps"]
drug_enzymes = artifacts["drug_enzymes"]
drug_cyps = artifacts["drug_cyps"]
drug_roles = artifacts["drug_roles"]
drug_sideeffects = artifacts["drug_sideeffects"]
drug_atc = artifacts["drug_atc"]

print("Artifacts loaded.")
print(artifacts.keys())

Artifacts loaded.
dict_keys(['drug_fps', 'drug_enzymes', 'drug_cyps', 'drug_roles', 'drug_sideeffects', 'drug_atc'])


In [10]:
def jaccard(a, b):
    a = set(a) if a is not None else set()
    b = set(b) if b is not None else set()
    if len(a) == 0 and len(b) == 0:
        return 0.0
    return len(a & b) / len(a | b)

def shared_count(a, b):
    a = set(a) if a is not None else set()
    b = set(b) if b is not None else set()
    return len(a & b)

def get_set_feature(dictionary, drug_id):
    return dictionary.get(drug_id, set())

def get_atc_levels(drug_id):
    atc = drug_atc.get(drug_id, {})
    return [
        atc.get("niveau_1", None),
        atc.get("niveau_2", None),
        atc.get("niveau_3", None),
        atc.get("niveau_4", None)
    ]

def atc_match_features(d1, d2):
    atc1 = get_atc_levels(d1)
    atc2 = get_atc_levels(d2)
    return [1 if a == b and a is not None else 0 for a, b in zip(atc1, atc2)]

def build_pair_features_level2(row):
    d1 = row["Drug1"]
    d2 = row["Drug2"]

    fp1 = drug_fps.get(d1)
    fp2 = drug_fps.get(d2)

    if fp1 is None or fp2 is None:
        return None

    fp1 = np.array(fp1, dtype=np.float32)
    fp2 = np.array(fp2, dtype=np.float32)

    morgan_features = np.concatenate([fp1, fp2])

    enz1 = get_set_feature(drug_enzymes, d1)
    enz2 = get_set_feature(drug_enzymes, d2)

    cyp1 = get_set_feature(drug_cyps, d1)
    cyp2 = get_set_feature(drug_cyps, d2)

    role1 = get_set_feature(drug_roles, d1)
    role2 = get_set_feature(drug_roles, d2)

    se1 = get_set_feature(drug_sideeffects, d1)
    se2 = get_set_feature(drug_sideeffects, d2)

    extra_features = np.array([
        jaccard(enz1, enz2),
        shared_count(enz1, enz2),

        jaccard(cyp1, cyp2),
        shared_count(cyp1, cyp2),

        jaccard(role1, role2),

        jaccard(se1, se2),
        shared_count(se1, se2),

        *atc_match_features(d1, d2)
    ], dtype=np.float32)

    return np.concatenate([morgan_features, extra_features]).astype(np.float32)

In [11]:
max_per_class = 1000

df_level2_sample = (
    df_level2
    .groupby("Interaction_ID", group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), max_per_class), random_state=42))
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("Sample shape:", df_level2_sample.shape)
print("Classes:", df_level2_sample["Interaction_ID"].nunique())
print(df_level2_sample["Interaction_ID"].value_counts().describe())

Sample shape: (39242, 7)
Classes: 61
count      61.000000
mean      643.311475
std       355.391312
min       104.000000
25%       329.000000
50%       735.000000
75%      1000.000000
max      1000.000000
Name: count, dtype: float64


C:\Users\user\AppData\Local\Temp\ipykernel_18976\2008225998.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), max_per_class), random_state=42))


In [12]:
X_list = []
valid_indices = []

for idx, row in df_level2_sample.iterrows():
    features = build_pair_features_level2(row)
    if features is not None:
        X_list.append(features)
        valid_indices.append(idx)

X_level2 = np.vstack(X_list).astype(np.float32)
df_level2_valid = df_level2_sample.loc[valid_indices].copy()

print("X_level2 shape:", X_level2.shape)
print("Data shape:", df_level2_valid.shape)
print("Memory GB:", round(X_level2.nbytes / 1e9, 3))

gc.collect()

X_level2 shape: (39236, 4107)
Data shape: (39236, 7)
Memory GB: 0.645


0

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le_level2 = LabelEncoder()

y_level2 = le_level2.fit_transform(df_level2_valid["Interaction_ID"])

print("Number of encoded Level2 classes:", len(le_level2.classes_))
print("Encoded labels:", y_level2[:10])

X_train, X_test, y_train, y_test = train_test_split(
    X_level2,
    y_level2,
    test_size=0.2,
    random_state=42,
    stratify=y_level2
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Number of encoded Level2 classes: 61
Encoded labels: [57 19 46 41 47 26 48 14 55 35]
X_train: (31388, 4107)
X_test: (7848, 4107)
y_train: (31388,)
y_test: (7848,)


In [14]:
from xgboost import XGBClassifier

level2_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(le_level2.classes_),
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

level2_model.fit(X_train, y_train)

print("Level 2 model trained.")

Level 2 model trained.


In [12]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix, top_k_accuracy_score

y_pred = level2_model.predict(X_test)
y_proba = level2_model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")
top3 = top_k_accuracy_score(y_test, y_proba, k=3)

print("Level 2 Accuracy:", acc)

print("Level 2 Balanced Accuracy:", bal_acc)
print("Level 2 Macro F1:", f1_macro)
print("Level 2 Weighted F1:", f1_weighted)
print("Level 2 Top-3 Accuracy:", top3)

Level 2 Accuracy: 0.9260958205912334
Level 2 Balanced Accuracy: 0.9448304513974595
Level 2 Macro F1: 0.939766054585794
Level 2 Weighted F1: 0.9238040002068892
Level 2 Top-3 Accuracy: 0.9843272171253823


In [17]:
y_pred = level2_model.predict(X_test)
y_proba = level2_model.predict_proba(X_test)

In [21]:
import os

files_to_check = [
    "Level2_XGB_Model_min100_sample1000.pkl",
    "Level2_LabelEncoder.pkl",
    "Level2_Metrics.csv",
    "Level2_Classification_Report.csv",
    "X_test_Level2.npy",
    "y_test_Level2.npy",
    "y_pred_Level2.npy",
    "y_proba_Level2.npy"
]

for f in files_to_check:
    print(f, "exists:", os.path.exists(f))

Level2_XGB_Model_min100_sample1000.pkl exists: True
Level2_LabelEncoder.pkl exists: True
Level2_Metrics.csv exists: False
Level2_Classification_Report.csv exists: False
X_test_Level2.npy exists: True
y_test_Level2.npy exists: True
y_pred_Level2.npy exists: True
y_proba_Level2.npy exists: True


In [22]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    top_k_accuracy_score
)

acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")
top3 = top_k_accuracy_score(y_test, y_proba, k=3)

metrics_level2 = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Macro F1",
        "Weighted F1",
        "Top-3 Accuracy"
    ],
    "Value": [
        acc,
        bal_acc,
        f1_macro,
        f1_weighted,
        top3
    ]
})

metrics_level2.to_csv("Level2_Metrics.csv", index=False)

report = classification_report(
    y_test,
    y_pred,
    target_names=[str(c) for c in le_level2.classes_],
    output_dict=True
)

report_df = pd.DataFrame(report).T
report_df.to_csv("Level2_Classification_Report.csv", index=True)

print("Saved: Level2_Metrics.csv")
print("Saved: Level2_Classification_Report.csv")

display(metrics_level2)
display(report_df.head(20))

Saved: Level2_Metrics.csv
Saved: Level2_Classification_Report.csv


,Metric,Value
0,Accuracy,0.926096
1,Balanced Accuracy,0.944830
2,Macro F1,0.939766
3,Weighted F1,0.923804
4,Top-3 Accuracy,0.984327


,precision,recall,f1-score,support
2,0.980296,0.995000,0.987593,200.0
3,0.814371,0.680000,0.741144,200.0
5,0.957317,0.987421,0.972136,159.0
6,0.955556,0.788991,0.864322,109.0
10,0.918660,0.960000,0.938875,200.0
11,0.975490,1.000000,0.987593,199.0
12,0.971014,0.957143,0.964029,70.0
13,0.956522,0.990000,0.972973,200.0
14,1.000000,1.000000,1.000000,28.0
15,0.986486,1.000000,0.993197,146.0


In [23]:
import os

files_to_check = [
    "Level2_XGB_Model_min100_sample1000.pkl",
    "Level2_LabelEncoder.pkl",
    "Level2_Metrics.csv",
    "Level2_Classification_Report.csv",
    "X_test_Level2.npy",
    "y_test_Level2.npy",
    "y_pred_Level2.npy",
    "y_proba_Level2.npy"
]

for f in files_to_check:
    print(f, "exists:", os.path.exists(f))

Level2_XGB_Model_min100_sample1000.pkl exists: True
Level2_LabelEncoder.pkl exists: True
Level2_Metrics.csv exists: True
Level2_Classification_Report.csv exists: True
X_test_Level2.npy exists: True
y_test_Level2.npy exists: True
y_pred_Level2.npy exists: True
y_proba_Level2.npy exists: True


In [1]:
import pandas as pd

input_file = "all_id_interaction_PK_PD.csv"

# نقرأ الملف على أجزاء لتفادي الذاكرة
pk_samples = []
pd_samples = []

target_n = 1000

for chunk in pd.read_csv(input_file, chunksize=200000):
    chunk = chunk[chunk["PK_PD"].isin(["PK", "PD"])].copy()
    
    pk_chunk = chunk[chunk["PK_PD"] == "PK"]
    pd_chunk = chunk[chunk["PK_PD"] == "PD"]
    
    if len(pk_samples) < target_n and len(pk_chunk) > 0:
        pk_samples.append(pk_chunk.sample(n=min(len(pk_chunk), 300), random_state=42))
    
    if len(pd_samples) < target_n and len(pd_chunk) > 0:
        pd_samples.append(pd_chunk.sample(n=min(len(pd_chunk), 300), random_state=42))
    
    pk_total = sum(len(x) for x in pk_samples)
    pd_total = sum(len(x) for x in pd_samples)
    
    if pk_total >= target_n and pd_total >= target_n:
        break

pk_df = pd.concat(pk_samples).sample(n=target_n, random_state=42)
pd_df = pd.concat(pd_samples).sample(n=target_n, random_state=42)

demo_df = pd.concat([pk_df, pd_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# التطبيق يحتاج Drug1 و Drug2 فقط
app_demo = demo_df.rename(columns={
    "Drug1 ID": "Drug1",
    "Drug2 ID": "Drug2"
})[["Drug1", "Drug2", "PK_PD", "Interaction_clean"]]

app_demo.to_csv("DDI_application_sample_2000_pairs.csv", index=False)

print(app_demo.shape)
print(app_demo["PK_PD"].value_counts())
display(app_demo.head())

(2000, 4)
PK_PD
PD    1000
PK    1000
Name: count, dtype: int64


,Drug1,Drug2,PK_PD,Interaction_clean
0,DB01156,DB13496,PD,the risk or severity of seizure can be increas...
1,DB01248,DB15762,PK,the serum concentration of (.*) can be decreas...
2,DB11235,DB13160,PD,the risk or severity of adverse effects can be...
3,DB06282,DB06212,PK,(.*) may increase the excretion rate of (.*) w...
4,DB00715,DB11598,PD,the risk or severity of hemorrhage can be incr...


In [3]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ملف نتائج التطبيق
pred = pd.read_csv("ddi_predictions (2).csv")

# نأخذ فقط الصفوف التي نجح فيها التطبيق
pred_success = pred[pred["Level1_prediction"].notna()].copy()

# الملف الأصلي الذي فيه الحقيقة PK/PD
true_df = pd.read_csv(
    "all_id_interaction_PK_PD.csv",
    usecols=["Drug1 ID", "Drug2 ID", "PK_PD"]
)

true_df = true_df[true_df["PK_PD"].isin(["PK", "PD"])].copy()

true_df = true_df.rename(columns={
    "Drug1 ID": "Drug1_ID",
    "Drug2 ID": "Drug2_ID",
    "PK_PD": "True_PK_PD"
})

# إزالة التكرار
true_df = true_df.drop_duplicates(subset=["Drug1_ID", "Drug2_ID"])

# الدمج بين نتائج التطبيق والحقيقة
comparison = pred_success.merge(
    true_df,
    on=["Drug1_ID", "Drug2_ID"],
    how="left"
)

print("Successful predictions:", len(pred_success))
print("Rows with true labels:", comparison["True_PK_PD"].notna().sum())

comparison_valid = comparison[comparison["True_PK_PD"].notna()].copy()

# التقييم
acc = accuracy_score(
    comparison_valid["True_PK_PD"],
    comparison_valid["Level1_prediction"]
)

print("Application comparison Accuracy:", acc)

print("\nConfusion matrix:")
print(confusion_matrix(
    comparison_valid["True_PK_PD"],
    comparison_valid["Level1_prediction"],
    labels=["PK", "PD"]
))

print("\nClassification report:")
print(classification_report(
    comparison_valid["True_PK_PD"],
    comparison_valid["Level1_prediction"],
    labels=["PK", "PD"]
))

# تقييم التنبؤات عالية الثقة فقط
for threshold in [0.60, 0.70, 0.80]:
    subset = comparison_valid[
        comparison_valid["Level1_confidence"] >= threshold
    ]
    
    if len(subset) > 0:
        acc_th = accuracy_score(
            subset["True_PK_PD"],
            subset["Level1_prediction"]
        )
        print(
            f"Confidence >= {threshold}: "
            f"N = {len(subset)}, Accuracy = {acc_th:.3f}"
        )

comparison_valid.to_csv("DDI_application_comparison_results.csv", index=False)

Successful predictions: 909
Rows with true labels: 909
Application comparison Accuracy: 0.6567656765676567

Confusion matrix:
[[403 173]
 [139 194]]

Classification report:
              precision    recall  f1-score   support

          PK       0.74      0.70      0.72       576
          PD       0.53      0.58      0.55       333

    accuracy                           0.66       909
   macro avg       0.64      0.64      0.64       909
weighted avg       0.66      0.66      0.66       909

Confidence >= 0.6: N = 661, Accuracy = 0.717
Confidence >= 0.7: N = 452, Accuracy = 0.796
Confidence >= 0.8: N = 232, Accuracy = 0.836


In [4]:
import pickle
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

ARTIFACTS_FILE = "DDI_feature_artifacts.pkl"
DRUGBANK_SMILES_FILE = "drugbank_smiles.csv"

OUTPUT_ARTIFACTS = "DDI_feature_artifacts_extended.pkl"
OUTPUT_DRUG_LIST = "Drug_list_for_app_extended.csv"

# =========================
# Load original artifacts
# =========================

with open(ARTIFACTS_FILE, "rb") as f:
    artifacts = pickle.load(f)

drug_fps = artifacts["drug_fps"]
drug_enzymes = artifacts["drug_enzymes"]
drug_cyps = artifacts["drug_cyps"]
drug_roles = artifacts["drug_roles"]
drug_sideeffects = artifacts["drug_sideeffects"]
drug_atc = artifacts["drug_atc"]

print("Original drugs in fingerprints:", len(drug_fps))

# =========================
# Load DrugBank SMILES file
# =========================

df = pd.read_csv(DRUGBANK_SMILES_FILE)
df.columns = [c.strip() for c in df.columns]

required_cols = ["drugbank_id", "name", "smiles"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")

df["drugbank_id"] = df["drugbank_id"].astype(str).str.strip()
df["name"] = df["name"].astype(str).str.strip()
df["smiles"] = df["smiles"].astype(str).str.strip()

df = df[
    df["drugbank_id"].notna() &
    df["smiles"].notna() &
    (df["smiles"] != "") &
    (df["smiles"].str.lower() != "nan")
].copy()

df = df.drop_duplicates(subset=["drugbank_id"])

print("Drugs with SMILES in file:", len(df))

# =========================
# Morgan fingerprint function
# =========================

def smiles_to_morgan(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        return None
    
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=n_bits
    )
    
    arr = np.zeros((n_bits,), dtype=np.float32)
    for i in range(n_bits):
        arr[i] = fp.GetBit(i)
    
    return arr

# =========================
# Extend artifacts
# =========================

added = 0
already_exists = 0
failed = 0

for _, row in df.iterrows():
    drug_id = row["drugbank_id"]
    smiles = row["smiles"]
    
    if drug_id in drug_fps:
        already_exists += 1
        continue
    
    fp = smiles_to_morgan(smiles)
    
    if fp is None:
        failed += 1
        continue
    
    drug_fps[drug_id] = fp
    
    # Biological annotations are unavailable for newly added drugs
    drug_enzymes.setdefault(drug_id, set())
    drug_cyps.setdefault(drug_id, set())
    drug_roles.setdefault(drug_id, set())
    drug_sideeffects.setdefault(drug_id, set())
    drug_atc.setdefault(drug_id, {})
    
    added += 1

print("Already existed:", already_exists)
print("Added new drugs:", added)
print("Failed SMILES:", failed)
print("New drugs in fingerprints:", len(drug_fps))

# =========================
# Save extended artifacts
# =========================

artifacts["drug_fps"] = drug_fps
artifacts["drug_enzymes"] = drug_enzymes
artifacts["drug_cyps"] = drug_cyps
artifacts["drug_roles"] = drug_roles
artifacts["drug_sideeffects"] = drug_sideeffects
artifacts["drug_atc"] = drug_atc

with open(OUTPUT_ARTIFACTS, "wb") as f:
    pickle.dump(artifacts, f)

print("Saved:", OUTPUT_ARTIFACTS)

# =========================
# Save extended drug list for app
# =========================

drug_list = df[["drugbank_id", "name", "smiles"]].copy()
drug_list = drug_list.rename(columns={
    "drugbank_id": "DrugBank_ID",
    "name": "Name",
    "smiles": "SMILES"
})

drug_list = drug_list[
    drug_list["DrugBank_ID"].isin(drug_fps.keys())
].copy()

drug_list.to_csv(OUTPUT_DRUG_LIST, index=False)

print("Saved:", OUTPUT_DRUG_LIST)
print("Drug list size:", len(drug_list))

Original drugs in fingerprints: 2385
Drugs with SMILES in file: 8288


[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerator
[16:48:55] DEPRECATION WARNING: please use MorganGenerat

Already existed: 1686
Added new drugs: 6596
Failed SMILES: 6
New drugs in fingerprints: 8981
Saved: DDI_feature_artifacts_extended.pkl
Saved: Drug_list_for_app_extended.csv
Drug list size: 8282


[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerator
[16:49:10] DEPRECATION WARNING: please use MorganGenerat

In [5]:
import pandas as pd

input_file = "all_id_interaction_PK_PD.csv"
output_file = "DDI_known_lookup.csv.gz"

usecols = ["Drug1 ID", "Drug2 ID", "PK_PD", "Interaction_clean"]

df = pd.read_csv(input_file, usecols=usecols)

df = df[df["PK_PD"].isin(["PK", "PD"])].copy()

df = df.rename(columns={
    "Drug1 ID": "Drug1_ID",
    "Drug2 ID": "Drug2_ID",
    "PK_PD": "Known_PK_PD",
    "Interaction_clean": "Known_interaction"
})

df["Drug1_ID"] = df["Drug1_ID"].astype(str).str.strip()
df["Drug2_ID"] = df["Drug2_ID"].astype(str).str.strip()

# key بنفس ترتيب الملف
df["pair_key"] = df["Drug1_ID"] + "||" + df["Drug2_ID"]

# key بدون اعتبار الترتيب للبحث العام
df["unordered_key"] = df.apply(
    lambda r: "||".join(sorted([r["Drug1_ID"], r["Drug2_ID"]])),
    axis=1
)

# إزالة التكرارات
df = df.drop_duplicates(subset=["pair_key"])

df.to_csv(output_file, index=False, compression="gzip")

print("Saved:", output_file)
print("Rows:", len(df))
print(df["Known_PK_PD"].value_counts())

Saved: DDI_known_lookup.csv.gz
Rows: 1355205
Known_PK_PD
PD    837560
PK    517645
Name: count, dtype: int64
